# B2-022-probabilistic-latent-models — Practice p12 — Solution

**Type:** constrained-coding · **Difficulty:** core · **Concepts:** variational-autoencoder

*50 minutes.*  
**Set:** B  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260927`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:F6-svd-spectral`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C9-dimensionality-reduction`, `book1:C11-neural-training`, `B2-021-cross-modal-transformers-vision`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:F6-svd-spectral](../../../../book1/units/F6-svd-spectral/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C9-dimensionality-reduction](../../../../book1/units/C9-dimensionality-reduction/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb).

## Supplied cell (copied verbatim from the statement)

In [ ]:
import torch

x = torch.tensor([[1.0, 0.0, -1.0, 2.0], [0.5, 0.5, 0.5, 0.5], [-2.0, 1.0, 0.0, 1.0]])
x_hat = torch.tensor([[0.8, 0.1, -1.0, 1.5], [0.5, 0.0, 1.0, 0.5], [-1.0, 1.0, 0.5, 0.0]])
mu = torch.tensor([[0.0, 0.0], [0.5, -0.5], [1.0, 2.0]])
logvar = torch.tensor([[0.0, 0.0], [-1.0, 0.0], [0.5, -0.5]])
ATOL = 1e-6
RTOL = 1e-6

## Solution

With a unit-variance Gaussian decoder, $-\log p(x\mid z)=\tfrac12\sum_j(x_j-\hat x_j)^2+\tfrac d2\log2\pi$; dropping the constant gives the reconstruction term **summed over features**.
The KL to $N(0,I)$ is **summed over latents**.
The per-example negative ELBO is `recon + kl`, and the batch loss is its **mean over the batch**, which is the same as `recon.mean() + kl.mean()` by linearity and is invariant to duplicating every row.

In [ ]:
def _check_inputs(x, x_hat, mu, logvar):
    for name, t in (("x", x), ("x_hat", x_hat), ("mu", mu), ("logvar", logvar)):
        if t.dim() != 2:
            raise ValueError(f"{name} must be 2-D")
    if x.shape != x_hat.shape:
        raise ValueError("x and x_hat shapes differ")
    if mu.shape != logvar.shape:
        raise ValueError("mu and logvar shapes differ")
    if x.shape[0] != mu.shape[0]:
        raise ValueError("batch sizes differ")


def negative_elbo_terms(x, x_hat, mu, logvar):
    _check_inputs(x, x_hat, mu, logvar)
    recon = 0.5 * ((x - x_hat) ** 2).sum(dim=1)
    kl = 0.5 * (mu ** 2 + logvar.exp() - logvar - 1).sum(dim=1)
    return recon, kl


def negative_elbo(x, x_hat, mu, logvar):
    recon, kl = negative_elbo_terms(x, x_hat, mu, logvar)
    return (recon + kl).mean()


recon, kl = negative_elbo_terms(x, x_hat, mu, logvar)
loss = negative_elbo(x, x_hat, mu, logvar)
loss_doubled = negative_elbo(x.repeat_interleave(2, dim=0), x_hat.repeat_interleave(2, dim=0),
                             mu.repeat_interleave(2, dim=0), logvar.repeat_interleave(2, dim=0))

leaf_mu = mu.clone().requires_grad_(True)
leaf_logvar = logvar.clone().requires_grad_(True)
leaf_x_hat = x_hat.clone().requires_grad_(True)
negative_elbo(x, leaf_x_hat, leaf_mu, leaf_logvar).backward()

def variant_feature_mean(x, x_hat, mu, logvar):
    recon = 0.5 * ((x - x_hat) ** 2).mean(dim=1)
    kl = 0.5 * (mu ** 2 + logvar.exp() - logvar - 1).sum(dim=1)
    return (recon + kl).mean()

def variant_kl_subtracted(x, x_hat, mu, logvar):
    recon, kl = negative_elbo_terms(x, x_hat, mu, logvar)
    return (recon - kl).mean()

va = variant_feature_mean(x, x_hat, mu, logvar)
vb = variant_kl_subtracted(x, x_hat, mu, logvar)
print(recon, kl, loss.item(), va.item(), vb.item())

### Answer check

In [ ]:
assert torch.allclose(recon, torch.tensor([0.15, 0.25, 1.125]), atol=ATOL, rtol=RTOL)
assert torch.allclose(kl, torch.tensor([0.0, 0.4339397, 2.6276259]), atol=ATOL, rtol=RTOL)
assert torch.allclose(loss, torch.tensor(1.5288552), atol=ATOL, rtol=RTOL)
assert recon.dtype == torch.float32 and kl.dtype == torch.float32
assert loss.shape == () and recon.shape == (3,) and kl.shape == (3,)
assert torch.allclose(loss, recon.mean() + kl.mean(), atol=ATOL, rtol=RTOL)
assert torch.allclose(loss_doubled, loss, atol=ATOL, rtol=RTOL)
for leaf in (leaf_mu, leaf_logvar, leaf_x_hat):
    assert leaf.grad is not None and bool(torch.isfinite(leaf.grad).all())
assert abs(va.item() - loss.item()) > 0.1
assert abs(vb.item() - loss.item()) > 0.1